# DiMEx model pipeline (Colab)

Trains and evaluates the valence-arousal regressors.

**Stage 1** learns the valence-arousal scale on DimABSA (SemEval-2026 Task 3, English restaurant and laptop).  
**Stage 2** adapts to drug reviews on Druglib; the validation slice is cut from Druglib's train file only, and the official test split stays held out.  
**Evaluate** scores the Druglib test split and Drugs.com (star ratings for satisfaction, plus a hand-labeled sample for effectiveness and side effects).

Before starting: *Runtime → Change runtime type → GPU* (T4 works; A100 or L4 is faster and supports bf16).

Checkpoints and results are saved to Google Drive, so you can run one model per session and pick up later.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/DiMEx'
WORK = '/content/drive/MyDrive/dimex'          # checkpoints, results and data live here
os.makedirs(WORK, exist_ok=True)

if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/av-jones/DiMEx.git {REPO}
%cd {REPO}
!pip -q install -r requirements-train.txt

## 2. Data

**DimABSA** is cloned from the task's GitHub repo.  
**Druglib**: download the UCI Druglib zip, unzip it, and put `drugLibTrain_raw.tsv` and `drugLibTest_raw.tsv` in `My Drive/dimex/data/druglib/`.  
**Drugs.com**: put `drugsComTest_raw.tsv` (Kaggle mirror) in `My Drive/dimex/data/drugscom/`.  
**Hand labels** (optional until you make them): `My Drive/dimex/data/drugscom_handlabels.csv` with columns `text, effectiveness, side_effects` (1-5 each).

In [ ]:
import glob
DATA = f'{WORK}/data'
if not os.path.exists('/content/DimABSA2026'):
    !git clone -q https://github.com/DimABSA/DimABSA2026.git /content/DimABSA2026

# English, Track A, labeled training files. CHECK this list: it should show restaurant and laptop train files.
DIMABSA = sorted(f for f in glob.glob('/content/DimABSA2026/**/*.jsonl', recursive=True)
                 if 'eng' in f.lower() and 'train' in f.lower() and 'stance' not in f.lower())
print('DimABSA files:', *DIMABSA, sep='\n  ')

DRUGLIB = f'{DATA}/druglib'
DRUGSCOM = f'{DATA}/drugscom/drugsComTest_raw.tsv'
HANDLABELS = f'{DATA}/drugscom_handlabels.csv'
for p in [DRUGLIB, DRUGSCOM, HANDLABELS]:
    print('found  ' if os.path.exists(p) else 'MISSING', p)

## 3. Smoke test (about 2 minutes)
A tiny run with 200 examples and one epoch, to catch problems before a long run.

In [ ]:
!python -m training.run stage1 --model DeBERTa-v3 --dimabsa {' '.join(DIMABSA)} \
    --out /content/smoke/stage1 --epochs 1 --limit 200
!python -m training.run stage2 --init /content/smoke/stage1 --druglib {DRUGLIB} \
    --out /content/smoke/stage2 --epochs 1 --limit 200

## 4. Full run for one model
Pick a model from: `DeBERTa-v3`, `ModernBERT`, `RoBERTa`, `PubMedBERT`, `BioBERT`, `ClinicalBERT`.  
Each model takes roughly 20-60 minutes on a T4, depending on its size and the DimABSA data size.

In [ ]:
MODEL = 'DeBERTa-v3'
SEED = 42
BATCH = 16          # lower to 8 (and set GRAD_ACCUM = 2) if you run out of GPU memory
GRAD_ACCUM = 1

slug = MODEL.lower().replace('-', '').replace('_', '')
CK = f'{WORK}/ckpts/{slug}/seed{SEED}'

!python -m training.run stage1 --model {MODEL} --dimabsa {' '.join(DIMABSA)} --out {CK}/stage1 \
    --epochs 4 --batch_size {BATCH} --grad_accum {GRAD_ACCUM} --seed {SEED}
!python -m training.run stage2 --init {CK}/stage1 --druglib {DRUGLIB} --out {CK}/stage2 \
    --epochs 6 --batch_size {BATCH} --grad_accum {GRAD_ACCUM} --seed {SEED}

## 5. Evaluate

In [ ]:
extra = ''
if os.path.exists(DRUGSCOM):   extra += f' --drugscom {DRUGSCOM}'
if os.path.exists(HANDLABELS): extra += f' --handlabels {HANDLABELS}'
!python -m training.run evaluate --ckpt {CK}/stage2 --druglib {DRUGLIB} {extra} \
    --out {WORK}/results/{slug}_seed{SEED}.csv

## 6. Ablation: does DimABSA pretraining help?
Same model trained on Druglib only, skipping stage 1. Comparing it with section 4 is a result for the paper.

In [ ]:
CK_ABL = f'{WORK}/ckpts/{slug}_nostage1/seed{SEED}'
!python -m training.run stage2 --init {MODEL} --druglib {DRUGLIB} --out {CK_ABL}/stage2 \
    --epochs 6 --batch_size {BATCH} --grad_accum {GRAD_ACCUM} --seed {SEED}
!python -m training.run evaluate --ckpt {CK_ABL}/stage2 --druglib {DRUGLIB} {extra} \
    --out {WORK}/results/{slug}_nostage1_seed{SEED}.csv

## 7. Compare all results so far

In [ ]:
import pandas as pd
files = sorted(glob.glob(f'{WORK}/results/*.csv'))
res = pd.concat([pd.read_csv(f).assign(run=os.path.basename(f)[:-4]) for f in files], ignore_index=True)
table = res.pivot_table(index='run', columns=['dataset', 'aspect'], values=['MAE', 'RMSE'])
display(table.round(3))
res.to_csv(f'{WORK}/results/all_results.csv', index=False)

## 8. Push a model to the Hugging Face Hub
Then set its `ckpt` in `core/registry.py` on GitHub, and the app uses the real model.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()
HF_REPO = 'YOUR_HF_USERNAME/dimex-' + slug
!python -m training.run push --ckpt {CK}/stage2 --repo {HF_REPO}